# SOAP map of Al–O–N structures

This notebook shows how to:
- load a set of atomistic structures and corresponding DFT energies,
- compute SOAP descriptors for each structure,
- build a **similarity kernel** between structures,
- project that similarity information into two dimensions with **Kernel PCA**,
- inspect selected structures directly from the 2D map,
- color the map using a simple energy-based quantity.

The aim is not only to make the code run, but also to understand what each step is doing.


In [ ]:
# Basic notebook setup
# `%matplotlib inline` is useful when interactive plotting backends are not available.
%matplotlib inline

from matplotlib import pylab as plt
import nglview
import ipywidgets as ipw
from IPython.display import display, clear_output

import random as rnd
from operator import itemgetter

from ase.io import read
import numpy as np

from rascal.representations import SphericalInvariants
from rascal.models import Kernel


## 1. Load the structures and their DFT energies

We read a trajectory containing many Al–O–N structures and the corresponding total energies.

We then:
1. switch on periodic boundary conditions,
2. wrap the atomic coordinates back into the simulation cell.

Wrapping is useful because two equivalent periodic structures may otherwise look different if some atoms
sit just outside the unit cell.


In [ ]:
# Load the full trajectory
framestot = read("traj.xyz", ":")

# Apply periodic boundary conditions and wrap atoms inside the cell.
for frame in framestot:
    frame.pbc = True
    frame.wrap(eps=1e-11)

# Load the corresponding total energies.
entotfull = np.loadtxt("entotsorted.dat")

print(f"Loaded {len(framestot)} structures.")
print(f"Loaded {len(entotfull)} energies.")


## 2. Select a random subset of structures

For a first exploration, it is often convenient to work on a smaller subset instead of the full dataset.

In [ ]:
# Select a random subset of structures
seed = 7
rnd.seed(seed)

N = 50
idrnd = np.asarray(rnd.sample(list(range(len(framestot))), N))

# Extract the chosen structures and the matching energies.
frames = list(itemgetter(*idrnd.tolist())(framestot))
entot = np.asarray(list(itemgetter(*idrnd.tolist())(entotfull)))

print(f"Selected {len(frames)} structures out of {len(framestot)}.")


## 3. Build a structure map with SOAP + kernel PCA

The idea is the following:

- **SOAP** converts each atomic structure into a numerical descriptor.
- A **kernel** converts those descriptors into a similarity matrix.
- **Kernel PCA** projects that similarity information into a low-dimensional map.

Nearby points in the final map should correspond to structures that are similar according to the chosen SOAP kernel.


In [ ]:
# SOAP hyperparameters
hypers = dict(
    soap_type="PowerSpectrum",
    interaction_cutoff=3.5,
    max_radial=6,
    max_angular=6,
    gaussian_sigma_constant=0.4,
    gaussian_sigma_type="Constant",
    cutoff_smooth_width=0.5,
)

# Build the SOAP descriptor and a cosine kernel between whole structures.
soap = SphericalInvariants(**hypers)
kernel = Kernel(
    soap,
    name="Cosine",
    zeta=2,
    target_type="Structure",
    kernel_type="Full",
)


The kernel compares structures after SOAP has encoded their local atomic environments.

Because we use `target_type='Structure'`, the final kernel compares **whole structures** rather than single atoms.


In [ ]:
# Compute the SOAP descriptors for all selected frames.
managers = soap.transform(frames)


In [ ]:
# Build the global similarity matrix K.
# K[i, j] is the similarity between structures i and j.
Kmat = kernel(managers)

print("Kernel matrix shape:", Kmat.shape)


In [ ]:
# For a normalized cosine kernel, diagonal entries should be 1 (or extremely close to 1).
Kmat[1][1]


In [ ]:
print("Number of selected structures:", len(frames))


In [ ]:
from sklearn.decomposition import KernelPCA


In [ ]:
# Fit Kernel PCA directly on the precomputed similarity matrix.
kpca = KernelPCA(n_components=2, kernel="precomputed")
kpca.fit(Kmat)


In [ ]:
# Project the structures into a 2D space for visualization.
X = kpca.transform(Kmat)


In [ ]:
print("Projected coordinates shape:", X.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X[:, 0], X[:, 1], s=12)
ax.set_xlabel("KPCA component 1")
ax.set_ylabel("KPCA component 2")
ax.set_title("SOAP structure map")
fig.tight_layout()
plt.show()


## 4. Inspect a selected structure directly on the map

The original direct-visualization cell used `ase.visualize.view(..., viewer='ngl')` and then manually removed widget components.
That approach can work in some environments, but it is more fragile than necessary.

Below, we use a cleaner `nglview` workflow:
- one helper function draws the selected point on the 2D map,
- another helper builds an `nglview` widget for the chosen structure.

This is easier to read and usually more robust inside notebooks.


In [ ]:
def plot_selected_frame(frame_index, coords):
    """Plot the 2D map and highlight a selected structure."""
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(coords[:, 0], coords[:, 1], s=12, alpha=0.7)
    ax.scatter(coords[frame_index, 0], coords[frame_index, 1], s=50, c="red")
    ax.set_xlabel("KPCA component 1")
    ax.set_ylabel("KPCA component 2")
    ax.set_title(f"Selected structure: frame {frame_index}")
    fig.tight_layout()
    plt.show()


def make_structure_widget(frame):
    """Create an NGL widget for an ASE Atoms object."""
    widget = nglview.show_ase(frame)
    widget.clear_representations()
    widget.add_ball_and_stick(aspectRatio=2.0, opacity=1.0)
    widget.add_unitcell()
    widget.center()
    return widget


# Example:
# frame = 50
# plot_selected_frame(frame, X)
# display(make_structure_widget(frames[frame]))


In [ ]:
# Interactive slider to inspect structures one by one.
plot_out = ipw.Output()
view_out = ipw.Output()

idf = ipw.IntSlider(
    value=7,
    min=0,
    max=N - 1,
    step=1,
    description="frame:",
    continuous_update=False,
)

display(idf, plot_out, view_out)


def on_frame_change(change):
    frame_index = idf.value

    with plot_out:
        clear_output(wait=True)
        plot_selected_frame(frame_index, X)

    with view_out:
        clear_output(wait=True)
        try:
            display(make_structure_widget(frames[frame_index]))
        except Exception as exc:
            print("Could not display the NGL widget in this environment.")
            print(f"Reason: {exc}")


# Show one structure immediately when the cell is run, then update on slider changes.
on_frame_change(None)
idf.observe(on_frame_change, names="value")


## 5. Color the map with an energy-related quantity

We now compute a simple energy quantity using pure AlN and pure Al$_2$O$_3$ structures as references,
and then use it to color the 2D map.


## 5a. Count the number of Al, N, and O atoms in each structure

The helper function below returns an array with:
`[n_Al, n_N, n_O]`


In [ ]:
def getnatms(frm):
    """Return the numbers of Al, N, and O atoms in one structure."""
    atomic_numbers = frm.get_atomic_numbers()
    return np.asarray([
        np.count_nonzero(atomic_numbers == 13),  # Al
        np.count_nonzero(atomic_numbers == 7),   # N
        np.count_nonzero(atomic_numbers == 8),   # O
    ])


## 5b. Identify reference structures for AlN and Al$_2$O$_3$

We look for:
- structures with no oxygen → treated here as pure AlN,
- structures with no nitrogen → treated here as pure Al$_2$O$_3$.

According to the dataset note:
- the AlN supercell contains 96 formula units,
- the Al$_2$O$_3$ supercell contains 48 formula units.


In [ ]:
# Get atomic counts for the full dataset
natms = np.asarray([getnatms(frame) for frame in framestot])

# Identify reference structures
idxALN = np.where(natms[:, 2] == 0)    # no oxygen
idxAL2O3 = np.where(natms[:, 1] == 0)  # no nitrogen

# Reference energies per formula unit
enALN = entotfull[idxALN][0] / 96.0
enAL2O3 = entotfull[idxAL2O3][0] / 48.0

print("Reference energy for AlN (per formula unit):", enALN)
print("Reference energy for Al2O3 (per formula unit):", enAL2O3)


In [ ]:
# Atomic counts only for the selected subset used in the SOAP map
natms = np.asarray([getnatms(frame) for frame in frames])


## 5c. Compute the cohesive energy per atom

We use the expression

$(E_i - n_N\,\mu_{\mathrm{AlN}} - n_O\,\mu_{\mathrm{Al_2O_3}}/3) / n_{\mathrm{atoms}}$.

Here the goal is mainly comparative: this quantity helps visualize how the energetic landscape varies across the structure map.


In [ ]:
def get_encoes(etot, ealn, eal2o3, atms):
    """Compute the energy quantity used to color the map."""
    return (etot - atms[1] * ealn - atms[2] * eal2o3 / 3.0) / atms.sum()


In [ ]:
# Compute the energy quantity for the selected subset
ecoes = np.asarray([
    get_encoes(en, enALN, enAL2O3, natms[i]) for i, en in enumerate(entot)
])


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(X[:, 0], X[:, 1], s=12, c=ecoes)
ax.set_xlabel("KPCA component 1")
ax.set_ylabel("KPCA component 2")
ax.set_title("SOAP map colored by cohesive energy")
fig.colorbar(sc, ax=ax, label="Cohesive energy per atom")
fig.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sc = ax.scatter(X[:, 0], X[:, 1], s=12, c=natms[:, 2])
ax.set_xlabel("KPCA component 1")
ax.set_ylabel("KPCA component 2")
ax.set_title("SOAP map colored by number of O atoms")
fig.colorbar(sc, ax=ax, label="Number of O atoms")
fig.tight_layout()
plt.show()
